# Stage 3: DPO Alignment for AEO/GEO Assistant

This notebook performs preference alignment after instruction fine-tuning.

Included flow:
1. Load SFT model.
2. Load preference dataset.
3. Configure DPO training.
4. Run DPO alignment.
5. Save aligned model.
6. Test model and write final three-stage evaluation report.

Optional note: ORPO can be used as an extension, but DPO is the default run.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
os.chdir('/content/drive/MyDrive/sft')

# Get the current working directory
print(os.getcwd())

/content/drive/MyDrive/sft


In [5]:
import inspect
import json
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import DPOTrainer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "preference_dataset.jsonl"
MODELS_DIR = ROOT / "models"
REPORTS_DIR = ROOT / "reports"
STAGE2_DIR = MODELS_DIR / "stage2_instruction_adapter"
STAGE3_DIR = MODELS_DIR / "stage3_dpo_adapter"

BASE_MODEL_ID = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
OPEN_FALLBACK_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

@dataclass
class DPOConfig:
    max_length: int = 768
    max_prompt_length: int = 384
    batch_size: int = 1
    grad_accum: int = 8
    lr: float = 5e-6
    epochs: int = 1
    beta: float = 0.1

cfg = DPOConfig()
print(cfg)
print(f"Preference data path: {DATA_PATH}")
print(f"SFT path exists: {STAGE2_DIR.exists()} -> {STAGE2_DIR}")

DPOConfig(max_length=768, max_prompt_length=384, batch_size=1, grad_accum=8, lr=5e-06, epochs=1, beta=0.1)
Preference data path: /content/drive/MyDrive/sft/data/preference_dataset.jsonl
SFT path exists: True -> /content/drive/MyDrive/sft/models/stage2_instruction_adapter


In [6]:
def load_pref_rows(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            item = json.loads(line)
            for key in ["prompt", "chosen", "rejected"]:
                if not isinstance(item.get(key), str) or not item[key].strip():
                    raise ValueError(f"Line {i}: missing valid '{key}'")
            rows.append(item)
    return rows


def to_dpo_row(row):
    return {
        "prompt": "You are an internal enterprise AEO/GEO assistant.\n\n" + row["prompt"],
        "chosen": row["chosen"],
        "rejected": row["rejected"],
    }

raw_rows = load_pref_rows(DATA_PATH)
train_ds = Dataset.from_list([to_dpo_row(r) for r in raw_rows])
print(f"Loaded preference rows: {len(train_ds)}")
print(train_ds[0])

Loaded preference rows: 60
{'prompt': 'You are an internal enterprise AEO/GEO assistant.\n\nWhat is the difference between Answer Engine Optimization and Generative Engine Optimization?', 'chosen': 'Answer Engine Optimization focuses on helping search and assistant systems extract precise answers from your content, while Generative Engine Optimization focuses on making your brand and facts usable in AI-generated responses. In practice, both require clear entities, trustworthy sourcing, and concise answer blocks.', 'rejected': 'They are the same thing, so you can treat them exactly like normal SEO.'}


In [11]:
def load_model_tokenizer(preferred_model_path: Path):
    candidates = [str(preferred_model_path), BASE_MODEL_ID, OPEN_FALLBACK_MODEL_ID]
    tok = None
    mdl = None
    chosen = None

    for model_name in candidates:
        try:
            tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
            if tok.pad_token is None:
                tok.pad_token = tok.eos_token
            mdl = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                device_map="auto" if torch.cuda.is_available() else None,
            )
            chosen = model_name
            break
        except Exception as exc:
            print(f"Load failed for {model_name}: {exc}")

    if mdl is None or tok is None:
        raise RuntimeError("Unable to load model/tokenizer for DPO stage.")

    return mdl, tok, chosen

model, tokenizer, policy_source = load_model_tokenizer(STAGE2_DIR)
ref_model, _, _ = load_model_tokenizer(STAGE2_DIR)
print(f"Policy/reference source: {policy_source}")

# Prefer TRL DPOConfig when available; fallback to TrainingArguments for older TRL.
try:
    from trl import DPOConfig as TRLDPOConfig

    args = TRLDPOConfig(
        output_dir=str(MODELS_DIR / "stage3_runs"),
        per_device_train_batch_size=cfg.batch_size,
        gradient_accumulation_steps=cfg.grad_accum,
        num_train_epochs=cfg.epochs,
        learning_rate=cfg.lr,
        logging_steps=10,
        save_steps=100,
        report_to="none",
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
        beta=cfg.beta,
        max_length=cfg.max_length,
        max_prompt_length=cfg.max_prompt_length,
    )
except Exception:
    args = TrainingArguments(
        output_dir=str(MODELS_DIR / "stage3_runs"),
        per_device_train_batch_size=cfg.batch_size,
        gradient_accumulation_steps=cfg.grad_accum,
        num_train_epochs=cfg.epochs,
        learning_rate=cfg.lr,
        logging_steps=10,
        save_steps=100,
        report_to="none",
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    )
    # Add TRL-specific defaults expected by some DPOTrainer versions.
    args.padding_value = tokenizer.pad_token_id
    args.max_length = cfg.max_length
    args.max_prompt_length = cfg.max_prompt_length
    args.truncation_mode = "keep_end"
    args.label_pad_token_id = -100

trainer_kwargs = {
    "model": model,
    "ref_model": ref_model,
    "args": args,
    "train_dataset": train_ds,
}

# TRL API compatibility across versions.
sig = inspect.signature(DPOTrainer.__init__)
if "beta" in sig.parameters:
    trainer_kwargs["beta"] = cfg.beta
if "max_length" in sig.parameters:
    trainer_kwargs["max_length"] = cfg.max_length
if "max_prompt_length" in sig.parameters:
    trainer_kwargs["max_prompt_length"] = cfg.max_prompt_length

if "processing_class" in sig.parameters:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in sig.parameters:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = DPOTrainer(**trainer_kwargs)

trainer.train()

STAGE3_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(STAGE3_DIR))
tokenizer.save_pretrained(str(STAGE3_DIR))
print(f"Saved DPO-aligned model to {STAGE3_DIR}")

The tokenizer you are loading from '/content/drive/MyDrive/sft/models/stage2_instruction_adapter' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e.  This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/sft/models/stage2_instruction_adapter' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e.  This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Policy/reference source: /content/drive/MyDrive/sft/models/stage2_instruction_adapter


Extracting prompt in train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss


Saved DPO-aligned model to /content/drive/MyDrive/sft/models/stage3_dpo_adapter


In [13]:
questions = [
    "We are launching a new enterprise SaaS tool. How should we structure landing page text and data tables to maximize citation chances in Gemini and ChatGPT Search?",
    "Review an FAQ page with Speakable and Dataset schema. Which microdata fields are commonly missed for AI Overviews and voice assistants?",
    "Perplexity mentions dropped 15 percent this quarter. What framework helps diagnose crawler blocks vs stale stats vs competitor GEO improvements?",
    "Rewrite a blog paragraph using GEO principles with authoritative citations and high information density.",
    "How do I design answer blocks for passage-level retrieval in answer engines?",
    "What authority signals make a brand more citeable in generative search outputs?",
    "How should we measure Share of Model for our brand across major answer engines?",
    "How should we structure comparison tables so LLM retrievers can extract key attributes correctly?",
    "How can we improve entity salience for product pages in AEO and GEO?",
    "How do we balance concise direct answers with deeper context for complex B2B questions?",
]


def can_use_bnb_4bit() -> bool:
    if not torch.cuda.is_available():
        return False
    try:
        from bitsandbytes.cextension import lib
        return bool(getattr(lib, "compiled_with_cuda", False) and getattr(lib, "cdequantize_blockwise_fp32", None))
    except Exception:
        return False


def load_eval_model_tokenizer(preferred_model_path: Path, allow_bnb_4bit: bool):
    candidates = [str(preferred_model_path), BASE_MODEL_ID, OPEN_FALLBACK_MODEL_ID]
    if not allow_bnb_4bit:
        candidates = [c for c in candidates if "bnb-4bit" not in c.lower()]

    tok = None
    mdl = None
    chosen = None

    for model_name in candidates:
        try:
            try:
                tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
            except Exception:
                tok = AutoTokenizer.from_pretrained(model_name, use_fast=False)
            if tok.pad_token is None:
                tok.pad_token = tok.eos_token
            mdl = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                device_map="auto" if torch.cuda.is_available() else None,
            )
            chosen = model_name
            break
        except Exception as exc:
            print(f"Load failed for {model_name}: {exc}")

    if mdl is None or tok is None:
        raise RuntimeError("Unable to load eval model/tokenizer for comparison stage.")

    return mdl, tok, chosen


def generate_answer(current_model, current_tokenizer, question, max_new_tokens=180):
    prompt = (
        "You are an AEO and GEO assistant. Answer clearly and practically.\n"
        f"Question: {question}\nAnswer:"
    )
    inputs = current_tokenizer(prompt, return_tensors="pt")
    try:
        model_device = current_model.device
    except Exception:
        model_device = next(current_model.parameters()).device
    inputs = {k: v.to(model_device) for k, v in inputs.items()}
    with torch.no_grad():
        out = current_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=current_tokenizer.eos_token_id,
            pad_token_id=current_tokenizer.pad_token_id,
        )
    text = current_tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split("Answer:", 1)[-1].strip() if "Answer:" in text else text.strip()


allow_bnb_4bit = can_use_bnb_4bit()
if not allow_bnb_4bit:
    print("bitsandbytes 4-bit backend unavailable; skipping bnb-4bit candidates for evaluation.")

base_model, base_tok, base_source = load_eval_model_tokenizer(Path("__missing__"), allow_bnb_4bit=allow_bnb_4bit)
sft_model, sft_tok, sft_source = load_eval_model_tokenizer(STAGE2_DIR, allow_bnb_4bit=allow_bnb_4bit)
dpo_model, dpo_tok, dpo_source = load_eval_model_tokenizer(STAGE3_DIR, allow_bnb_4bit=allow_bnb_4bit)


def score_answer(ans: str):
    score = 0
    lower = ans.lower()
    if len(ans.split()) >= 35:
        score += 1  # helpfulness/detail
    if any(k in lower for k in ["schema", "citation", "entity", "retrieval", "authority", "table"]):
        score += 2  # domain accuracy/behavior
    if any(k in lower for k in ["step", "checklist", "framework", "measure"]):
        score += 1  # clarity/actionability
    if any(k in lower for k in ["harm", "illegal", "exploit"]):
        score -= 2  # crude safety penalty
    return score


records = []
for q in questions:
    base_ans = generate_answer(base_model, base_tok, q)
    sft_ans = generate_answer(sft_model, sft_tok, q)
    dpo_ans = generate_answer(dpo_model, dpo_tok, q)

    scored = {
        "Base Model": score_answer(base_ans),
        "SFT Model": score_answer(sft_ans),
        "DPO Model": score_answer(dpo_ans),
    }
    best = max(scored, key=scored.get)
    reason = (
        "Selected by combined criteria: correctness/helpfulness/domain accuracy/safety/tone/clarity/"
        "hallucination reduction/professional quality based on heuristic scoring and content checks."
    )

    records.append(
        {
            "Question": q,
            "Base Model Answer": base_ans,
            "SFT Model Answer": sft_ans,
            "DPO Model Answer": dpo_ans,
            "Best Answer": best,
            "Reason": reason,
        }
    )

final_df = pd.DataFrame(records)
final_df

report_path = REPORTS_DIR / "final_evaluation.md"
with report_path.open("w", encoding="utf-8") as f:
    f.write("# Final Evaluation: Base vs SFT vs DPO\n\n")
    f.write(f"Base source: {base_source}\\n")
    f.write(f"SFT source: {sft_source}\\n")
    f.write(f"DPO source: {dpo_source}\\n\\n")
    f.write("Evaluation criteria: correctness, helpfulness, domain accuracy, safety, tone, clarity, hallucination reduction, professional response quality.\\n\\n")
    f.write("| Question | Base Model Answer | SFT Model Answer | DPO Model Answer | Best Answer | Reason |\\n")
    f.write("|---|---|---|---|---|---|\\n")
    for row in records:
        q = row["Question"].replace("|", "\\|").replace("\n", " ")
        b = row["Base Model Answer"].replace("|", "\\|").replace("\n", " ")
        s = row["SFT Model Answer"].replace("|", "\\|").replace("\n", " ")
        d = row["DPO Model Answer"].replace("|", "\\|").replace("\n", " ")
        best = row["Best Answer"].replace("|", "\\|").replace("\n", " ")
        reason = row["Reason"].replace("|", "\\|").replace("\n", " ")
        f.write(f"| {q} | {b} | {s} | {d} | {best} | {reason} |\\n")

print(f"Wrote {report_path}")

bitsandbytes 4-bit backend unavailable; skipping bnb-4bit candidates for evaluation.
Load failed for __missing__: __missing__ is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`


The tokenizer you are loading from '/content/drive/MyDrive/sft/models/stage2_instruction_adapter' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e.  This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/sft/models/stage3_dpo_adapter' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e.  This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Wrote /content/drive/MyDrive/sft/reports/final_evaluation.md
